# Notebook A: LoRA SFT of Qwen2.5-1.5B-Instruct (Colab T4, default runtime)
Reads the curated data from the private HF dataset, trains with AlignTune (fp16, no bf16 on a T4),
pushes a checkpoint to HF every epoch, then pushes the adapter and the merged model.
Re-running resumes from the latest checkpoint on HF, or skips training if the final models are already there.
Needs a Colab secret named `HF_TOKEN` (notebook access on). SGLang is never installed here.

In [1]:
import os
os.environ["PURE_TRL_MODE"] = "1"  # AlignTune's switch: never import unsloth (it patches TRL globally)
!nvidia-smi | head -12
!pip install -q aligntune
import aligntune, trl, transformers, torch
print("aligntune", aligntune.__version__, "| trl", trl.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)

Sat Oct 24 16:16:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

aligntune 0.1.28 | trl 1.7.1 | transformers 5.20.2 | torch 2.11.0+cu130


In [2]:
from google.colab import userdata
from huggingface_hub import HfApi, login

HF_USER = "bharataggarwal56"
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add a Colab secret named HF_TOKEN and enable notebook access"
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
assert api.whoami()["name"] == HF_USER, "token belongs to a different account"

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_REPO = f"{HF_USER}/ticketforge-data"
TAG = "2026-10-24"
ADAPTER_REPO = f"{HF_USER}/ticketforge-qwen2.5-1.5b-lora"
MERGED_REPO = f"{HF_USER}/ticketforge-qwen2.5-1.5b-merged"

In [3]:
from huggingface_hub import snapshot_download
DATA_DIR = snapshot_download(DATA_REPO, repo_type="dataset", local_dir="/content/ticketforge-data",
                             allow_patterns=[f"train_{TAG}/*"], token=HF_TOKEN)
TRAIN_DIR = f"{DATA_DIR}/train_{TAG}"
!ls -R {TRAIN_DIR} | head -20

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

/content/ticketforge-data/train_2026-10-24:
checksums.txt
dataset_card.md
lexsi_provenance.json
manifest.json
README.md
rejected.jsonl
train
validation

/content/ticketforge-data/train_2026-10-24/train:
sft_messages.jsonl

/content/ticketforge-data/train_2026-10-24/validation:
sft_messages.jsonl


## Token lengths: pick `max_seq_length` so no assistant answer is truncated

In [4]:
import json, math
import numpy as np
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE)
totals, answers = [], []
for split in ("train", "validation"):
    for line in open(f"{TRAIN_DIR}/{split}/sft_messages.jsonl"):
        msgs = json.loads(line)["messages"]
        ids = tok.apply_chat_template(msgs, tokenize=True)
        ids = ids["input_ids"] if hasattr(ids, "keys") else ids
        ans = tok(msgs[-1]["content"] + "<|im_end|>\n", add_special_tokens=False)["input_ids"]
        assert ids[-len(ans):] == ans, "assistant answer is not the tail of the templated example"
        totals.append(len(ids)); answers.append(len(ans))

for name, v in (("full example", totals), ("assistant answer", answers)):
    print(f"{name:17s} n={len(v)} max={max(v)} p99={int(np.percentile(v, 99))} mean={np.mean(v):.1f}")
MAX_SEQ_LEN = math.ceil((max(totals) + 8) / 64) * 64
assert max(totals) <= MAX_SEQ_LEN
print("max_seq_length =", MAX_SEQ_LEN)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

full example      n=1499 max=361 p99=342 mean=306.7
assistant answer  n=1499 max=65 p99=60 mean=52.1
max_seq_length = 384


## Training config
Settings come from reading the AlignTune source (see `docs/plan.md` §4); `task_type="sft"` because AlignTune 0.1.28 rejects `chat_completion`:
- `precision="fp16"` sets `SFTConfig(fp16=True, bf16=False)` and loads the weights in float16.
- AlignTune derives `max_steps` from epochs, so "every epoch" means `save_interval = eval_interval = steps_per_epoch`.
- Hub options reach `SFTConfig` as extra kwargs; the assert after training confirms they were applied.

In [5]:
N_TRAIN = sum(1 for _ in open(f"{TRAIN_DIR}/train/sft_messages.jsonl"))
N_VAL = sum(1 for _ in open(f"{TRAIN_DIR}/validation/sft_messages.jsonl"))
assert (N_TRAIN, N_VAL) == (1349, 150), (N_TRAIN, N_VAL)

BATCH, GRAD_ACCUM, EPOCHS = 4, 2, 3
STEPS_PER_EPOCH = N_TRAIN // (BATCH * GRAD_ACCUM)
print("steps/epoch", STEPS_PER_EPOCH, "total steps", STEPS_PER_EPOCH * EPOCHS)

TRAIN_KW = dict(
    model_name=BASE,
    dataset_name=TRAIN_DIR,
    config_name="sft_messages",
    backend="trl",
    task_type="sft",  # AlignTune 0.1.28 DataManager rejects "chat_completion"
    messages_column="messages",
    # No chat_template kwarg: AlignTune treats any value without Jinja markers (like "auto") as an Unsloth
    # template name and imports unsloth, which patches TRL's SFTTrainer. Qwen's own template is used instead.
    output_dir="/content/outputs/sft",
    num_epochs=EPOCHS,
    batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4,
    max_seq_length=MAX_SEQ_LEN,
    precision="fp16",
    peft_enabled=True,
    lora_r=16, lora_alpha=32, lora_dropout=0.05,
    lora_target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    save_interval=STEPS_PER_EPOCH,
    eval_interval=STEPS_PER_EPOCH,
    eval_strategy="steps",
    save_strategy="steps",
    push_to_hub=True,
    hub_model_id=ADAPTER_REPO,
    hub_strategy="all_checkpoints",
    hub_token=HF_TOKEN,
    hub_private_repo=True,
)

steps/epoch 168 total steps 504


## Re-run state: skip if finished, resume if checkpoints exist on HF

In [6]:
import re, sys
def repo_files(repo):
    try:
        return api.list_repo_files(repo, repo_type="model")
    except Exception:
        return []

adapter_files, merged_files = repo_files(ADAPTER_REPO), repo_files(MERGED_REPO)
FINAL_DONE = "adapter_model.safetensors" in adapter_files and any(f.endswith(".safetensors") for f in merged_files)
ckpts = sorted({int(m.group(1)) for f in adapter_files if (m := re.match(r"checkpoint-(\d+)/", f))})
RESUME_DIR = None
if FINAL_DONE:
    print("Adapter and merged model already on HF; skipping training and merge.")
elif ckpts:
    RESUME_DIR = snapshot_download(ADAPTER_REPO, repo_type="model", token=HF_TOKEN,
                                   local_dir="/content/resume", allow_patterns=[f"checkpoint-{ckpts[-1]}/*"]) + f"/checkpoint-{ckpts[-1]}"
    print("Resuming from", RESUME_DIR)
else:
    print("Fresh run")

Fresh run


In [7]:
if not FINAL_DONE:
    # AlignTune calls trainer.train() without arguments; inject the resume path.
    from trl import SFTTrainer
    if RESUME_DIR and not getattr(SFTTrainer, "_resume_patched", False):
        _orig_train = SFTTrainer.train
        def _train(self, resume_from_checkpoint=None, **kw):
            return _orig_train(self, resume_from_checkpoint=resume_from_checkpoint or RESUME_DIR, **kw)
        SFTTrainer.train = _train
        SFTTrainer._resume_patched = True

    from aligntune.core.backend_factory import create_sft_trainer
    trainer = create_sft_trainer(**TRAIN_KW)
    assert "unsloth" not in sys.modules and not trl.SFTTrainer.__module__.startswith("unsloth"), (
        "unsloth got imported and patches TRL. Run `pip uninstall -y unsloth unsloth_zoo`, restart the runtime, and re-run.")
    result = trainer.train()
    print(result)

    args = trainer.trainer.args
    assert args.fp16 and not args.bf16, (args.fp16, args.bf16)
    assert args.push_to_hub and args.hub_model_id == ADAPTER_REPO and str(args.hub_strategy).endswith("all_checkpoints"), "hub options were not applied"
    assert args.max_length == MAX_SEQ_LEN
    print(trainer.evaluate())

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
168,0.132571,0.131992,0.125166,0.962102,412287.000000
336,0.100142,0.122716,0.114186,0.963045,823521.000000
504,0.108719,0.121696,0.107843,0.962944,1234851.000000
505,0.108719,0.121698,0.107844,0.962966,1237302.000000


{'training_time': 1032.0936226844788, 'final_loss': 0.24189866356330342, 'model_path': '/content/outputs/sft', 'steps': 505, 'epochs': 0, 'metrics': {'train_runtime': 1020.8667, 'train_samples_per_second': 3.957, 'train_steps_per_second': 0.495, 'total_flos': 1.0286524258194432e+16, 'train_loss': 0.24189866356330342, 'epoch': 2.9881656804733727}, 'task_type': 'sft'}


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
0.108719,0.121696,505,0.107843,0.962944,1237302.000000


[aligntune] 2026-10-24 16:40:37 WARNING aligntune.core.sft.trainer_base:221 - Could not compute quality metrics: 'NoneType' object has no attribute 'evaluate'


{'eval_loss': 0.12169575691223145, 'eval_entropy': 0.10784335591291126, 'eval_mean_token_accuracy': 0.9629444354458859, 'eval_num_tokens': 1237302.0, 'task_type': <TaskType.SFT: 'sft'>, 'eval_perplexity': 1.1294103860855103, 'qualitative_samples': []}


## Save, push the adapter, merge, push the merged model

In [8]:
if not FINAL_DONE:
    from aligntune.utils import load_finetuned_model
    adapter_dir = trainer.save_model(output_dir="/content/outputs/sft")
    api.create_repo(ADAPTER_REPO, repo_type="model", private=True, exist_ok=True)
    api.upload_folder(repo_id=ADAPTER_REPO, repo_type="model", folder_path="/content/outputs/sft",
                      ignore_patterns=["checkpoint-*", "runs/*", "*.pt", "optimizer.*"],
                      commit_message="Final LoRA adapter")

    model, tokenizer = load_finetuned_model(output_dir="/content/outputs/sft", base_model=BASE)
    model.save_pretrained("/content/outputs/merged"); tokenizer.save_pretrained("/content/outputs/merged")
    api.create_repo(MERGED_REPO, repo_type="model", private=True, exist_ok=True)
    api.upload_folder(repo_id=MERGED_REPO, repo_type="model", folder_path="/content/outputs/merged",
                      commit_message="Merged fp16 model")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Sanity check on the merged model (one validation ticket)

In [9]:
import json, torch
from transformers import AutoModelForCausalLM
m = AutoModelForCausalLM.from_pretrained(MERGED_REPO, torch_dtype=torch.float16, token=HF_TOKEN).to("cuda")
t = AutoTokenizer.from_pretrained(MERGED_REPO, token=HF_TOKEN)
row = json.loads(open(f"{TRAIN_DIR}/validation/sft_messages.jsonl").readline())["messages"]
prompt = t.apply_chat_template(row[:2], add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
out = m.generate(**prompt, max_new_tokens=128, do_sample=False)
text = t.decode(out[0][prompt["input_ids"].shape[1]:], skip_special_tokens=True)
print(text)
print("gold:", row[2]["content"])
print("parses:", set(json.loads(text)) == set(json.loads(row[2]["content"])))
print(f"https://huggingface.co/{ADAPTER_REPO}\nhttps://huggingface.co/{MERGED_REPO}")

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

{"category": "shipping", "priority": "low", "product": "Nimbus Router X2", "sentiment": "neutral", "summary": "Customer reports order not arrived for Nimbus Router X2; wants a human agent.", "needs_human": true}
gold: {"category": "shipping", "priority": "low", "product": "Nimbus Router X2", "sentiment": "neutral", "summary": "Customer reports order not arrived for Nimbus Router X2; wants a human agent.", "needs_human": true}
parses: True
https://huggingface.co/bharataggarwal56/ticketforge-qwen2.5-1.5b-lora
https://huggingface.co/bharataggarwal56/ticketforge-qwen2.5-1.5b-merged
